<a href="https://colab.research.google.com/github/subham-28/PyTorch/blob/main/RNN_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import pandas as pd
df=pd.read_csv("/content/100_Unique_QA_Dataset.csv")

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [54]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace(".","")
  text=text.replace(",","")
  text=text.replace("?","")
  text=text.replace("!","")
  text=text.replace("'","")

  return text.split()

In [55]:
#vocab
vocab={'<UNK>':0}

In [56]:
def build_vocab(row):
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])

  merged_tokens=tokenized_question+tokenized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)

In [57]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [58]:
len(vocab)

324

In [59]:
#convert words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [60]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [61]:
import torch
from torch.utils.data import Dataset, DataLoader

In [62]:
from IPython.utils import text
#custom dataset class
class CustomDataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,idx):
    numerical_question = text_to_indices(self.df.iloc[idx]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[idx]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)


In [63]:
dataset = CustomDataset(df,vocab)

In [64]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [65]:
for qs,ans in dataloader:
  print(qs,ans)

tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([[170]])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([[188]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([[99]])
tensor([[ 42, 137, 118,   3, 247,   5, 248]]) tensor([[249]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([[194]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([[54]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([[106]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]]

In [66]:
import torch.nn as nn

In [67]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn = nn.RNN(50,64,batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self,question):
    embedded_question=self.embedding(question)
    hidden,final=self.rnn(embedded_question)
    output=self.fc(final.squeeze(0))

    return output

In [68]:
learning_rate = 0.001
epochs = 20

In [69]:
model = SimpleRNN(len(vocab))

In [70]:
criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [71]:
#training loop
for epoch in range(epochs):
  total_epoch_loss=0
  for qs, ans in dataloader:
    #zero gradient
    optimizer.zero_grad()

    #forward pass
    y_pred=model(qs)

    #loss
    loss = criteria(y_pred,ans[0])

    #backward pass
    loss.backward()

    #update params
    optimizer.step()

    #print loss
    total_epoch_loss+=loss.item()
  print(f"Epoch: {epoch+1}/{epochs} Loss: {total_epoch_loss}")

Epoch: 1/20 Loss: 525.2553262710571
Epoch: 2/20 Loss: 457.82099056243896
Epoch: 3/20 Loss: 379.7217309474945
Epoch: 4/20 Loss: 316.58904814720154
Epoch: 5/20 Loss: 265.16727566719055
Epoch: 6/20 Loss: 217.2034398317337
Epoch: 7/20 Loss: 173.26561307907104
Epoch: 8/20 Loss: 135.94252848625183
Epoch: 9/20 Loss: 105.62291371822357
Epoch: 10/20 Loss: 81.76732876896858
Epoch: 11/20 Loss: 63.66590115427971
Epoch: 12/20 Loss: 50.32908846437931
Epoch: 13/20 Loss: 40.17120938003063
Epoch: 14/20 Loss: 32.60124784708023
Epoch: 15/20 Loss: 26.72486236691475
Epoch: 16/20 Loss: 22.06534907221794
Epoch: 17/20 Loss: 18.76411794871092
Epoch: 18/20 Loss: 15.846871346235275
Epoch: 19/20 Loss: 13.547756634652615
Epoch: 20/20 Loss: 11.735306292772293


In [72]:
def predict(model, question, threshold=0.5):

  #convert qs to nums
  numerical_question = text_to_indices(question, vocab)

  #convert to tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  #send to model
  output = model(question_tensor)

  #convert to prob
  probs = torch.nn.functional.softmax(output, dim=1)

  #find index of max prob
  value,index = torch.max(probs,dim=1)

  if value<threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [80]:
predict(model, "What is the capital of Japan")

tokyo
